In [1]:
import os
import random
import argparse
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
SEED = 69

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(seed=SEED)

In [4]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
BATCH_SIZE = 64
EPOCHS = 50 # adjust it
LR = 1e-3 # adjust it
WEIGHT_DECAY = 1e-5 # adjust it
HIDDEN_SIZE = 128 # adjust it
NUM_LAYERS = 2 # adjust it
DROPOUT = 0.3 # adjust it
MAX_LEN = 800 # adjust it
INPUT_FEATURES = 7 # acc_x,y,z + rot_w,x,y,z
MIXUP_PROB = 0.0 # adjust it
MIXUP_ALPHA = 0.4 # adjust it
NOISE_STD = 0.02 # adjust it
N_FOLDS = 5

In [6]:
def fft_feats(df, col="acc_norm", k=12):
    rows = []
    for sid, x in df.groupby("sequence_id", sort=False)[col]:
        x = x.values.astype(np.float32)
        if len(x) == 0:
            rows.append([sid, 0, 0.0, 0.0, 0.0])
            continue

        x = x - x.mean()
        sp = np.abs(np.fft.rfft(x))

        if sp.size <= 1:
            dom = 0
            band = np.zeros(k, dtype=np.float32)
        else:
            dom = int(np.argmax(sp[1:]) + 1)
            band = sp[1:1+k]
            if band.size < k:
                band = np.pad(band, (0, k - band.size))

        rows.append([sid, dom, float(band.mean()), float(band.max()), float(np.sum(band**2))])

    return pd.DataFrame(rows, columns=["sequence_id", f"{col}_fft_dom_bin", f"{col}_fft_mean", f"{col}_fft_max", f"{col}_fft_energy"])


In [7]:
import numpy as np
import pandas as pd

SENSOR_COLS = ["acc_x","acc_y","acc_z","rot_w","rot_x","rot_y","rot_z"]

def build_seq_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["acc_norm"] = np.sqrt(df["acc_x"]**2 + df["acc_y"]**2 + df["acc_z"]**2)
    df["rot_norm"] = np.sqrt(df["rot_w"]**2 + df["rot_x"]**2 + df["rot_y"]**2 + df["rot_z"]**2)
    
    fft_acc = fft_feats(df, col="acc_norm", k=12)
    fft_rot = fft_feats(df, col="rot_norm", k=12)

    cols = SENSOR_COLS + ["acc_norm", "rot_norm"]

    g = df.groupby("sequence_id", sort=False)

    feats = g[cols].agg(["mean","std","min","max","median"])
    feats.columns = [f"{c}_{stat}" for c, stat in feats.columns]

    q25 = g[cols].quantile(0.25).add_suffix("_q25")
    q75 = g[cols].quantile(0.75).add_suffix("_q75")
    feats = feats.join(q25).join(q75)

    for c in cols:
        feats[f"{c}_energy"] = g[c].apply(lambda x: float(np.mean(np.square(x.values))))

    feats["seq_len"] = g.size().astype(np.float32)
    
    df["jerk_norm"] = np.sqrt(
    df.groupby("sequence_id", sort=False)["acc_x"].diff().fillna(0)**2 +
    df.groupby("sequence_id", sort=False)["acc_y"].diff().fillna(0)**2 +
    df.groupby("sequence_id", sort=False)["acc_z"].diff().fillna(0)**2
    )
    jerk_feats = df.groupby("sequence_id", sort=False)["jerk_norm"].agg(["mean","std","max"]).reset_index()
    jerk_feats.columns = ["sequence_id","jerk_mean","jerk_std","jerk_max"]


    dfd = df.groupby("sequence_id", sort=False)[cols].diff().fillna(0.0)
    dfd["sequence_id"] = df["sequence_id"].values
    gd = dfd.groupby("sequence_id", sort=False)
    dfeats = gd[cols].agg(["mean","std","max"])
    dfeats.columns = [f"d{c}_{stat}" for c, stat in dfeats.columns]
    feats = feats.join(dfeats)

    feats = feats.reset_index()
    feats = feats.merge(jerk_feats, on="sequence_id", how="left")  # ты jerk_feats посчитал, но не мерджил!
    feats = feats.merge(fft_acc, on="sequence_id", how="left")
    feats = feats.merge(fft_rot, on="sequence_id", how="left")
    return feats

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score

TRAIN_PATH = "aikc-idas-courses-2025-final-competition/train.csv"
TEST_PATH  = "aikc-idas-courses-2025-final-competition/test.csv"
SUB_PATH   = "aikc-idas-courses-2025-final-competition/sample_submission.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
sub = pd.read_csv(SUB_PATH)

y_seq = train_df.groupby("sequence_id", sort=False)["gesture"].first().reset_index()
subj_seq = train_df.groupby("sequence_id", sort=False)["subject_id"].first().reset_index()

le = LabelEncoder()
y_seq["y"] = le.fit_transform(y_seq["gesture"].values)

tr_feat = build_seq_features(train_df).merge(subj_seq, on="sequence_id", how="left").merge(y_seq[["sequence_id","y"]], on="sequence_id", how="left")
te_feat = build_seq_features(test_df) 

drop_cols = ["sequence_id","subject_id","y"]
X = tr_feat.drop(columns=drop_cols)
y = tr_feat["y"].values
groups = tr_feat["subject_id"].values

X_test = te_feat.drop(columns=["sequence_id"])


In [ ]:
def add_corr_feats(df):
    g = df.groupby("sequence_id", sort=False)
    def corr(a, b):
        return g.apply(lambda x: float(np.corrcoef(x[a].values, x[b].values)[0,1]) if x.shape[0] > 2 else 0.0)
    out = pd.DataFrame({
        "sequence_id": g.size().index,
        "corr_acc_xy": corr("acc_x","acc_y").values,
        "corr_acc_xz": corr("acc_x","acc_z").values,
        "corr_acc_yz": corr("acc_y","acc_z").values,
        "corr_rot_wx": corr("rot_w","rot_x").values,
        "corr_rot_wy": corr("rot_w","rot_y").values,
        "corr_rot_wz": corr("rot_w","rot_z").values,
    }).reset_index(drop=True)
    return out



In [ ]:
from xgboost import XGBClassifier

classes, counts = np.unique(y, return_counts=True)
cw = {c: (len(y) / (len(classes) * cnt)) for c, cnt in zip(classes, counts)}
sample_w = np.array([cw[yy] for yy in y], dtype=np.float32)

N_FOLDS = 5
gkf = GroupKFold(n_splits=N_FOLDS)

oof = np.zeros((len(y), len(classes)), dtype=np.float32)
test_proba = np.zeros((len(X_test), len(classes)), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups=groups), 1):
    Xtr, Xva = X.iloc[tr_idx], X.iloc[va_idx]
    ytr, yva = y[tr_idx], y[va_idx]

    wtr = sample_w[tr_idx]

    model = XGBClassifier(
    n_estimators=3000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    objective="multi:softprob",
    num_class=len(classes),
    tree_method="hist",
    random_state=42 + fold
    )

    model.fit(Xtr, ytr, sample_weight=wtr, verbose=False)


    pva = model.predict_proba(Xva)
    oof[va_idx] = pva
    pred_va = pva.argmax(axis=1)
    f1 = f1_score(yva, pred_va, average="macro")
    print(f"fold {fold}: macroF1={f1:.4f}")

    test_proba += model.predict_proba(X_test) / N_FOLDS

oof_pred = oof.argmax(axis=1)
print("OOF macroF1:", f1_score(y, oof_pred, average="macro"))


In [ ]:
test_pred = test_proba.argmax(axis=1)

sub_out = pd.read_csv(SUB_PATH)
sub_out["gesture"] = le.inverse_transform(test_pred)
sub_out.to_csv("submission.csv", index=False)
print("saved submission.csv", sub_out.shape)
sub_out.head()


In [ ]:
sample = pd.read_csv("aikc-idas-courses-2025-final-competition/sample_submission.csv")
assert set(sub.columns) == set(sample.columns)
assert len(sub) == len(sample)
assert set(sub["sequence_id"]) == set(sample["sequence_id"])
